# Логистическая регрессия — простыми словами

## Зачем это вообще нужно

Представь, что тебе нужно написать функцию, которая отвечает не "да" или "нет", а **"на сколько процентов да"**. Например:

```ts
function isSpam(email: Email): number {
  // вернуть не boolean, а вероятность от 0 до 1
  // 0.92 значит "на 92% уверен, что это спам"
}
```

Вот именно этим и занимается **логистическая регрессия**. Это алгоритм, который:
1. Смотрит на входные данные (признаки — features)
2. Считает **вероятность** (число от 0 до 1) того, что объект относится к какому-то классу
3. Если вероятность ≥ 50% → говорит "1" (класс есть, например "это спам")
4. Если < 50% → говорит "0" (класса нет)

Несмотря на слово "регрессия" в названии — это алгоритм для **классификации** (для задач вида "да/нет", "спам/не спам", "кот/не кот").

> 🇬🇧 На английском: *logistic regression* также называют *logit regression*. Слово **logit** — это сокращение от **log**istic un**it**, встретишь его в коде библиотек (PyTorch, TensorFlow) очень часто.

---

## Формула 1: как считается вероятность

$$
\hat{p} = h_\theta(\mathbf{x}) = \sigma(\boldsymbol{\theta}^T \mathbf{x})
$$

Смысл в двух словах: **сначала считаем взвешенную сумму признаков (как в обычной линейной регрессии), а потом "сжимаем" результат в диапазон от 0 до 1.**

Если думать в терминах кода:

```ts
function predict(x: number[], theta: number[]): number {
  const weightedSum = dotProduct(theta, x); // θᵀx
  return sigmoid(weightedSum);              // σ(...)
}
```

### Разбор символов

| Символ | Как читается | Что это такое | Аналогия из программирования |
|---|---|---|---|
| $\hat{p}$ | "пи с крышечкой" (p-hat) | Предсказанная вероятность (число от 0 до 1) | Значение, которое возвращает функция |
| $h_\theta(\mathbf{x})$ | "аш тета от икс" | "Гипотеза" (hypothesis) — сама модель как функция | Название функции: `model(x)` |
| $\sigma$ | "сигма" | Сигмоидная функция — "сжиматель" чисел в диапазон 0..1 | Как `Math.min/max`, но плавно |
| $\boldsymbol{\theta}$ | "тета" | Вектор параметров модели (веса) | `weights: number[]` |
| $\boldsymbol{\theta}^T$ | "тета транспонированная" | Тот же вектор, но повёрнутый для умножения | Технический шаг для матричного умножения |
| $\mathbf{x}$ | "икс" (жирным — вектор) | Вектор входных признаков одного объекта | `features: number[]` |
| $\boldsymbol{\theta}^T \mathbf{x}$ | "тета транспонированная на икс" | Скалярное произведение — взвешенная сумма | `dotProduct(theta, x)` |

> 🇯🇵 Кстати, "тета" (θ) по-японски произносится **シータ** (*shiita*) — просто транслитерация греческой буквы, никакого скрытого смысла.

---

## Формула 2: сигмоида — "сжиматель" в диапазон 0..1

$$
\sigma(t) = \frac{1}{1 + e^{-t}}
$$

Это и есть та самая функция, которая превращает **любое** число (хоть −1000, хоть +1000) в число **строго между 0 и 1**. Форма графика — красивая S-образная кривая:

![Сигмоидная функция](./fig-4-22.png)

Что видно на графике:
- Когда $t$ сильно отрицательный → результат стремится к **0**
- Когда $t = 0$ → результат ровно **0.5**
- Когда $t$ сильно положительный → результат стремится к **1**

На TypeScript эта формула выглядит совсем просто:

```ts
function sigmoid(t: number): number {
  return 1 / (1 + Math.exp(-t));
}
```

### Разбор символов

| Символ | Как читается | Что это такое |
|---|---|---|
| $\sigma(t)$ | "сигма от тэ" | Результат сигмоиды для числа $t$ |
| $t$ | "тэ" | Любое число (в нашем случае — это и есть $\theta^T x$ из формулы 1) |
| $e$ | "экспонента" / "число e" | Математическая константа ≈ 2.71828 (как $\pi$, только для экспоненты). В коде это `Math.E` или просто `Math.exp()` |
| $e^{-t}$ | "e в степени минус тэ" | Экспонента от $-t$. В коде: `Math.exp(-t)` |

> Не нужно понимать *почему* именно такая формула даёт S-образную кривую — важно запомнить: **это стандартный "конвертер" произвольного числа в вероятность**. Используется он не только в логистической регрессии, но и как функция активации в нейросетях.

---

## Формула 3: как из вероятности получить итоговый ответ (0 или 1)

$$
\hat{y} =
\begin{cases}
0 & \text{если } \hat{p} < 0.5 \\
1 & \text{если } \hat{p} \geq 0.5
\end{cases}
$$

Тут всё просто — это обычный `if`:

```ts
function classify(pHat: number): 0 | 1 {
  return pHat >= 0.5 ? 1 : 0;
}
```

### Разбор символов

| Символ | Как читается | Что это такое |
|---|---|---|
| $\hat{y}$ | "игрек с крышечкой" (y-hat) | Итоговое предсказание модели: класс 0 или 1 |
| $\hat{p}$ | "пи с крышечкой" | Вероятность из формулы 1 |

Порог 0.5 — это просто дефолт (можно поставить и другой, например 0.8, если хочешь быть более осторожным в предсказании "1").

> Кстати, значение $t = \theta^T x$ отдельно называют **logit** или **log-odds** — потому что это логарифм отношения "вероятность класса 1" к "вероятность класса 0". Знать это не обязательно, но термин будет часто попадаться в статьях.

---

## Формула 4: как модель понимает, что она ошиблась (Cost Function)

Чтобы обучить модель, нужен способ **измерить, насколько сильно она ошибается** — эта штука называется *cost function* (функция потерь/стоимости). Для одного примера она выглядит так:

$$
c(\boldsymbol{\theta}) =
\begin{cases}
-\log(\hat{p}) & \text{если } y = 1 \\
-\log(1-\hat{p}) & \text{если } y = 0
\end{cases}
$$

Смысл интуитивно: чем более уверенно модель ошиблась — тем сильнее её "штраф" (cost).

- Если правильный ответ **1**, а модель сказала $\hat{p}$ близко к **0** (то есть уверенно ошиблась) → штраф **огромный**
- Если правильный ответ **1**, а модель сказала $\hat{p}$ близко к **1** (угадала) → штраф **почти 0**
- Симметрично для случая, когда правильный ответ **0**

```ts
function cost(pHat: number, y: 0 | 1): number {
  return y === 1 ? -Math.log(pHat) : -Math.log(1 - pHat);
}
```

### Разбор символов

| Символ | Как читается | Что это такое |
|---|---|---|
| $c(\boldsymbol{\theta})$ | "цэ от тета" | Штраф (цена ошибки) при текущих параметрах $\theta$ |
| $y$ | "игрек" | **Настоящий**, правильный ответ (0 или 1) — то, что было в тренировочных данных |
| $\log$ | "логарифм" | Обычный натуральный логарифм. В коде — `Math.log()` |
| $-\log(\hat{p})$ | "минус логарифм от пи с крышечкой" | Штраф на случай, когда правильный ответ — 1 |

> Почему именно логарифм? У функции $-\log(t)$ есть удобное свойство: она **стремится к бесконечности**, когда $t \to 0$, и **стремится к 0**, когда $t \to 1$. То есть она идеально подходит, чтобы "сильно наказывать" уверенные, но неверные предсказания.

### Общая формула для всего датасета (Log Loss)

Формула выше — для одного примера. А чтобы посчитать общую ошибку модели по **всем** примерам сразу, их усредняют, и получается формула, известная как **log loss**:

$$
J(\boldsymbol{\theta}) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{p}^{(i)}) + (1-y^{(i)}) \log(1-\hat{p}^{(i)}) \right]
$$

Не пугайся — это просто **один и тот же `cost()` из примера выше, но применённый ко всем строкам датасета и усреднённый**:

```ts
function logLoss(data: { x: number[]; y: 0 | 1 }[], theta: number[]): number {
  const m = data.length;
  const total = data.reduce((sum, { x, y }) => {
    const pHat = predict(x, theta); // формула 1
    return sum + cost(pHat, y);     // формула 4 (в свёрнутом виде)
  }, 0);
  return total / m;
}
```

### Разбор символов

| Символ | Как читается | Что это такое | Аналогия из кода |
|---|---|---|---|
| $J(\boldsymbol{\theta})$ | "джей от тета" | Общая ошибка модели по всему датасету при параметрах $\theta$ | Возвращаемое значение `logLoss()` |
| $m$ | "эм" | Количество примеров (строк) в датасете | `data.length` |
| $\sum_{i=1}^{m}$ | "сумма от i равно 1 до эм" | Просуммировать что-то для каждого примера с 1-го по m-й | `data.reduce(...)` или `for` цикл |
| $y^{(i)}$ | "игрек и-тое" | Правильный ответ для **i-го** примера | `data[i].y` |
| $\hat{p}^{(i)}$ | "пи с крышечкой и-тое" | Предсказанная вероятность для **i-го** примера | `predict(data[i].x, theta)` |
| $-\frac{1}{m}$ | "минус один делить на эм" | Усреднение (деление на количество примеров) + смена знака | `total / m` (со знаком минус внутри суммы) |
| $i$ | "и" | Порядковый номер примера в датасете (индекс цикла) | `i` в `for (let i = 0; i < m; i++)` |
| $\log(\hat{p}^{(i)})$ | "логарифм от пи с крышечкой и-тое" | Логарифм предсказанной вероятности для i-го примера — "штрафная" часть на случай, если правильный ответ 1 | `Math.log(pHat)` |
| $(1-y^{(i)})$ | "один минус игрек и-тое" | Просто "переключатель": равен 0, если $y=1$, и равен 1, если $y=0$ | `y === 1 ? 0 : 1` |
| $\log(1-\hat{p}^{(i)})$ | "логарифм от один минус пи с крышечкой и-тое" | Логарифм "вероятности класса 0" — штрафная часть на случай, если правильный ответ 0 | `Math.log(1 - pHat)` |
| $[\,y^{(i)}\log(\hat p^{(i)}) + (1-y^{(i)})\log(1-\hat p^{(i)})\,]$ | "выражение в квадратных скобках" | Весь штраф для одного i-го примера — та же формула 4 (cost), просто записанная без if/else | `cost(pHat, y)` |

**Почему формула выглядит так, будто в ней сразу два слагаемых, хотя выше было "если/то"?** Это математический трюк: когда $y = 1$, второе слагаемое $(1-y)\log(1-\hat p)$ обнуляется само (умножается на 0), и остаётся только первое. И наоборот при $y=0$. Это просто способ записать `if/else` одной формулой без ветвления — программистский аналог тернарного оператора, применённого хитрым способом через умножение на 0/1.

---

## Формула 5: как модель обучается (градиент)

$$
\frac{\partial}{\partial \theta_j} J(\boldsymbol{\theta}) = \frac{1}{m} \sum_{i=1}^{m} \left( \sigma(\boldsymbol{\theta}^T \mathbf{x}^{(i)}) - y^{(i)} \right) x_j^{(i)}
$$

Это самая "страшная на вид" формула, но смысл у неё очень простой: **она говорит, в какую сторону и насколько сильно нужно подвинуть каждый параметр $\theta_j$, чтобы модель ошибалась чуть меньше.**

Логика такая:
1. Для каждого примера считаем **ошибку**: предсказание минус правильный ответ ($\sigma(\theta^Tx) - y$)
2. Умножаем эту ошибку на значение конкретного признака $x_j$
3. Усредняем по всем примерам
4. Получившееся число говорит: "увеличь / уменьши параметр $\theta_j$ вот на столько"

```ts
function gradientForParam(
  data: { x: number[]; y: 0 | 1 }[],
  theta: number[],
  j: number
): number {
  const m = data.length;
  const total = data.reduce((sum, { x, y }) => {
    const error = sigmoid(dotProduct(theta, x)) - y; // ошибка
    return sum + error * x[j];                       // умножаем на j-й признак
  }, 0);
  return total / m;
}
```

Дальше на каждой итерации обучения (это и есть **градиентный спуск**, gradient descent) параметры обновляются так:

```ts
theta[j] = theta[j] - learningRate * gradientForParam(data, theta, j);
```

И это повторяется много раз, пока ошибка (Log Loss из формулы 4) не станет достаточно маленькой.

### Разбор символов

| Символ | Как читается | Что это такое | Аналогия из кода |
|---|---|---|---|
| $\frac{\partial}{\partial \theta_j} J(\boldsymbol{\theta})$ | "частная производная джей по тета-жи" | Насколько сильно (и в какую сторону) изменится общая ошибка $J$, если чуть-чуть изменить именно параметр $\theta_j$ | Число, на которое надо скорректировать `theta[j]` |
| $\theta_j$ | "тета-жи" | j-й параметр (вес) модели, один конкретный элемент вектора $\theta$ | `theta[j]` |
| $\mathbf{x}^{(i)}$ | "икс и-тое" | Вектор признаков i-го примера | `data[i].x` |
| $x_j^{(i)}$ | "икс-жи и-тое" | j-й признак i-го примера (одно конкретное число) | `data[i].x[j]` |
| $\sigma(\boldsymbol{\theta}^T \mathbf{x}^{(i)}) - y^{(i)}$ | "сигма от тета-икс минус игрек" | Ошибка модели на i-м примере: предсказание минус правда | `predict(data[i].x, theta) - data[i].y` |

> 🇬🇧 "**Partial derivative**" (частная производная) — не нужно вникать глубоко в матан, чтобы использовать эту формулу. Достаточно понимать её как **"насколько изменится ошибка, если немного подкрутить один конкретный параметр"** — то есть это buildin-механизм автоматической настройки весов, без которого пришлось бы подбирать их руками.

---

## Итог: как всё это связано между собой

1. **Формула 1** ($\hat p = \sigma(\theta^Tx)$) — модель делает предсказание (вероятность)
2. **Формула 2** (сигмоида) — "сжимает" число в диапазон 0..1
3. **Формула 3** — превращает вероятность в чёткий ответ 0/1
4. **Формула 4** (Log Loss) — измеряет, насколько модель ошибается
5. **Формула 5** (градиент) — говорит, как подправить параметры $\theta$, чтобы ошибиться меньше в следующий раз

Весь процесс обучения — это цикл: **предсказать → посчитать ошибку → чуть-чуть подправить параметры → повторить**, пока ошибка не станет маленькой. По сути, ровно то же самое, что делает `while(loss > threshold) { updateWeights() }` в коде.